# Feature Engineering

In [3]:
import warnings
warnings.filterwarnings('ignore')

import re
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter
from rank_bm25 import BM25Okapi
from datasets import load_dataset

## Data Ingestion

In [8]:
dataset = load_dataset("ms_marco", "v2.1", split="validation")
marco_df = dataset.to_pandas()
marco_df.head()

,answers,passages,query,query_id,query_type,wellFormedAnswers
0,[A corporation is a company or group of people...,"{'is_selected': [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]...",. what is a corporation?,1102432,DESCRIPTION,[]
1,[Rachel Carson writes The Obligation to Endure...,"{'is_selected': [0, 0, 0, 0, 1, 1, 0, 0, 0, 0]...",why did rachel carson write an obligation to e...,1102431,DESCRIPTION,[]
2,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",why did the progressive movement fail to advan...,1102421,DESCRIPTION,[]
3,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",why do police need to understand what the fore...,1102315,DESCRIPTION,[]
4,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",do owls eat in the day,1101280,NUMERIC,[]


| Column | Type | Notes |
| :--- | :--- | :--- |
| query | string | The search query |
| passages | dict | Contains passage_text (list) and is_selected (list) |
| query_id | int | Unique query identifier |

In [10]:
marco_df.shape

(101093, 6)

Current size might be a bit too large for development purposes. Would be sub-sampling to 10,000 rows using random_state of 42 to ensure reproducibility

In [11]:
marco_df_samp = marco_df.sample(n=10000, random_state=42).reset_index(drop=True)